In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
folder = r"Data"

In [3]:
data = {}

for i in range(1, 9):
    function_folder = os.path.join(folder, f"function_{i}")

    X = np.load(os.path.join(function_folder, "initial_inputs.npy"))
    y = np.load(os.path.join(function_folder, "initial_outputs.npy"))

    columns = [f"x{j+1}" for j in range(X.shape[1])]

    df = pd.DataFrame(X, columns=columns)
    df["output"] = y

    data[i] = df

    print(f"Function {i}: {len(df)} observations, {X.shape[1]} dimensions")

Function 1: 10 observations, 2 dimensions
Function 2: 10 observations, 2 dimensions
Function 3: 15 observations, 3 dimensions
Function 4: 30 observations, 4 dimensions
Function 5: 20 observations, 4 dimensions
Function 6: 20 observations, 5 dimensions
Function 7: 30 observations, 6 dimensions
Function 8: 40 observations, 8 dimensions


In [4]:
new_points = {
    1: ([0.001243, 0.996919], 0),
    2: ([0.821778, 0.980301], -0.08204067029832242),
    3: ([0.409499, 0.399000, 0.519487], -0.02016690747473939),
    4: ([0.399157, 0.433203, 0.357699, 0.461233], -0.3006446531358864),
    5: ([0.352268, 0.854957, 0.929210, 0.999541], 2329.015853969669),
    6: ([0.910500, 0.039226, 0.999592, 0.081789, 0.939481], -2.661555742668831),
    7: ([0.120627, 0.403921, 0.379973, 0.146533, 0.335853, 0.763305], 1.9506122834052728),
    8: ([0.243397, 0.036115, 0.078358, 0.053721, 0.783854, 0.278393, 0.034947, 0.429032], 9.821986472796599)
}

for i in range(1, 9):

    X_new, y_new = new_points[i]

    new_row = pd.DataFrame(
        [X_new + [y_new]],
        columns=data[i].columns
    )

    data[i] = pd.concat(
        [data[i], new_row],
        ignore_index=True
    )

    print(f"Function {i}: {len(data[i])} observations")

Function 1: 11 observations
Function 2: 11 observations
Function 3: 16 observations
Function 4: 31 observations
Function 5: 21 observations
Function 6: 21 observations
Function 7: 31 observations
Function 8: 41 observations


In [5]:
for i in range(1, 9):

    df = data[i]

    # New observation is the final row
    new_output = df.iloc[-1]["output"]

    # All observations before this week's addition
    previous_data = df.iloc[:-1]

    previous_best = previous_data["output"].max()
    updated_best = df["output"].max()
    improvement = new_output - previous_best

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(f"Previous best: {previous_best:.6f}")
    print(f"New output:    {new_output:.6f}")
    print(f"Difference:    {improvement:.6f}")
    print(f"Updated best:  {updated_best:.6f}")

    if new_output > previous_best:
        print("Result: NEW BEST")
    else:
        print("Result: Did not improve")

    print()

FUNCTION 1
Previous best: 0.000000
New output:    0.000000
Difference:    -0.000000
Updated best:  0.000000
Result: Did not improve

FUNCTION 2
Previous best: 0.611205
New output:    -0.082041
Difference:    -0.693246
Updated best:  0.611205
Result: Did not improve

FUNCTION 3
Previous best: -0.034835
New output:    -0.020167
Difference:    0.014668
Updated best:  -0.020167
Result: NEW BEST

FUNCTION 4
Previous best: -4.025542
New output:    -0.300645
Difference:    3.724898
Updated best:  -0.300645
Result: NEW BEST

FUNCTION 5
Previous best: 1088.859618
New output:    2329.015854
Difference:    1240.156236
Updated best:  2329.015854
Result: NEW BEST

FUNCTION 6
Previous best: -0.714265
New output:    -2.661556
Difference:    -1.947291
Updated best:  -0.714265
Result: Did not improve

FUNCTION 7
Previous best: 1.364968
New output:    1.950612
Difference:    0.585644
Updated best:  1.950612
Result: NEW BEST

FUNCTION 8
Previous best: 9.598482
New output:    9.821986
Difference:    0.223

In [6]:
models_week2 = {}

for i in range(1, 9):

    df = data[i]

    # Separate inputs and outputs
    X = df.drop(columns="output").values
    y = df["output"].values

    # Same RBF kernel as Week 1
    kernel = RBF(
        length_scale=0.5,
        length_scale_bounds=(1e-5, 1e5)
    )

    # Fit Gaussian Process
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    gp.fit(X, y)

    models_week2[i] = gp

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)
    print("Fitted kernel:", gp.kernel_)
    print()

FUNCTION 1
Fitted kernel: RBF(length_scale=1e-05)

FUNCTION 2
Fitted kernel: RBF(length_scale=0.0453)

FUNCTION 3
Fitted kernel: RBF(length_scale=0.202)

FUNCTION 4
Fitted kernel: RBF(length_scale=0.522)

FUNCTION 5
Fitted kernel: RBF(length_scale=0.225)

FUNCTION 6
Fitted kernel: RBF(length_scale=0.481)

FUNCTION 7
Fitted kernel: RBF(length_scale=0.354)

FUNCTION 8
Fitted kernel: RBF(length_scale=0.913)



In [7]:
from sklearn.gaussian_process.kernels import Matern

# Function 1 data
df = data[1]

X = df.drop(columns="output").values
y = df["output"].values

# RBF model
rbf_kernel = RBF(
    length_scale=0.5,
    length_scale_bounds=(1e-5, 1e5)
)

rbf_gp = GaussianProcessRegressor(
    kernel=rbf_kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

rbf_gp.fit(X, y)


# Matérn model
matern_kernel = Matern(
    length_scale=0.5,
    length_scale_bounds=(1e-5, 1e5),
    nu=2.5
)

matern_gp = GaussianProcessRegressor(
    kernel=matern_kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

matern_gp.fit(X, y)


print("RBF kernel:")
print(rbf_gp.kernel_)

print("\nMatérn kernel:")
print(matern_gp.kernel_)

RBF kernel:
RBF(length_scale=1e-05)

Matérn kernel:
Matern(length_scale=1e-05, nu=2.5)


In [8]:
# Generate candidate points for Function 1
rng = np.random.default_rng(42)

X_candidates_f1 = rng.uniform(
    low=0,
    high=1,
    size=(100000, 2)
)

# Predictions from RBF
rbf_mean, rbf_std = rbf_gp.predict(
    X_candidates_f1,
    return_std=True
)

# Predictions from Matérn
matern_mean, matern_std = matern_gp.predict(
    X_candidates_f1,
    return_std=True
)

print("=" * 60)
print("FUNCTION 1 - RBF vs MATERN")
print("=" * 60)

print("\nRBF")
print(f"Predicted output range: {rbf_mean.min():.10f} to {rbf_mean.max():.10f}")
print(f"Uncertainty range:      {rbf_std.min():.10f} to {rbf_std.max():.10f}")

print("\nMATERN")
print(f"Predicted output range: {matern_mean.min():.10f} to {matern_mean.max():.10f}")
print(f"Uncertainty range:      {matern_std.min():.10f} to {matern_std.max():.10f}")

FUNCTION 1 - RBF vs MATERN

RBF
Predicted output range: -0.0003278239 to -0.0000001931
Uncertainty range:      0.0000355799 to 0.0010366701

MATERN
Predicted output range: -0.0003278239 to -0.0000003215
Uncertainty range:      0.0000459032 to 0.0010366701


In [9]:
# Store candidate points and results
week2_candidates = {}

rng = np.random.default_rng(42)

kappa = 2.5

for i in range(2, 9):

    df = data[i]
    gp = models_week2[i]

    n_dimensions = len(df.columns) - 1

    # Generate 100,000 candidate points
    X_candidates = rng.uniform(
        0,
        1,
        size=(100000, n_dimensions)
    )

    # Predict mean and uncertainty
    mean, std = gp.predict(
        X_candidates,
        return_std=True
    )

    # Calculate UCB
    ucb = mean + kappa * std

    # Select candidate with highest UCB
    best_index = np.argmax(ucb)
    selected_point = X_candidates[best_index]

    week2_candidates[i] = selected_point

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print("Selected point:", selected_point)
    print("Predicted output:", mean[best_index])
    print("Uncertainty:", std[best_index])
    print("UCB:", ucb[best_index])
    print()

FUNCTION 2
Selected point: [0.65528565 0.92989635]
Predicted output: 0.44107070745260113
Uncertainty: 0.19003228777907213
UCB: 0.9161514269002815

FUNCTION 3
Selected point: [0.39752993 0.09047714 0.44731895]
Predicted output: -0.05580113542024088
Uncertainty: 0.07430259032309242
UCB: 0.12995534038749015

FUNCTION 4
Selected point: [0.48075601 0.47400033 0.22905919 0.42917843]
Predicted output: -1.666506404136829
Uncertainty: 0.8868534178318199
UCB: 0.550627140442721

FUNCTION 5
Selected point: [0.51484244 0.85610509 0.96150603 0.99849219]
Predicted output: 2112.903795601483
Uncertainty: 303.2913666553336
UCB: 2871.132212239817

FUNCTION 6
Selected point: [0.24654859 0.34464495 0.47952502 0.733976   0.02533225]
Predicted output: -0.6645463641633735
Uncertainty: 0.3554831146702006
UCB: 0.224161422512128

FUNCTION 7
Selected point: [0.08033361 0.31746516 0.48551741 0.0555686  0.40688258 0.93197557]
Predicted output: 1.7729655691895485
Uncertainty: 0.24096833783176155
UCB: 2.3753864137689

In [10]:
for i in range(2, 9):

    df = data[i]

    # Find current best observation
    best_index = df["output"].idxmax()
    best_row = df.loc[best_index]

    best_X = best_row.drop("output").values.astype(float)
    best_y = best_row["output"]

    # Proposed Week 2 query
    new_X = week2_candidates[i]

    # Euclidean distance between proposed point and current best
    distance = np.linalg.norm(new_X - best_X)

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print("Current best point:")
    print(np.round(best_X, 6))

    print("Current best output:")
    print(round(best_y, 6))

    print("\nProposed new point:")
    print(np.round(new_X, 6))

    print("\nDistance from current best:")
    print(round(distance, 4))

    print()

FUNCTION 2
Current best point:
[0.702637 0.926564]
Current best output:
0.611205

Proposed new point:
[0.655286 0.929896]

Distance from current best:
0.0475

FUNCTION 3
Current best point:
[0.409499 0.399    0.519487]
Current best output:
-0.020167

Proposed new point:
[0.39753  0.090477 0.447319]

Distance from current best:
0.3171

FUNCTION 4
Current best point:
[0.399157 0.433203 0.357699 0.461233]
Current best output:
-0.300645

Proposed new point:
[0.480756 0.474    0.229059 0.429178]

Distance from current best:
0.1609

FUNCTION 5
Current best point:
[0.352268 0.854957 0.92921  0.999541]
Current best output:
2329.015854

Proposed new point:
[0.514842 0.856105 0.961506 0.998492]

Distance from current best:
0.1658

FUNCTION 6
Current best point:
[0.728186 0.154693 0.732552 0.693997 0.056401]
Current best output:
-0.714265

Proposed new point:
[0.246549 0.344645 0.479525 0.733976 0.025332]

Distance from current best:
0.5785

FUNCTION 7
Current best point:
[0.120627 0.403921 0.379

In [11]:
from scipy.spatial.distance import cdist

# Existing Function 1 observations
X_existing = data[1].drop(columns="output").values

# We already generated 100,000 candidates for Function 1
# in the RBF vs Matérn comparison
X_candidates = X_candidates_f1

# Distance from every candidate to every existing observation
distances = cdist(X_candidates, X_existing)

# Distance to nearest existing observation
nearest_distance = distances.min(axis=1)

# Candidate furthest from its nearest existing observation
best_index = np.argmax(nearest_distance)

function1_point = X_candidates[best_index]
function1_distance = nearest_distance[best_index]

print("=" * 60)
print("FUNCTION 1 - EXPLORATION")
print("=" * 60)

print("Selected point:")
print(function1_point)

print("\nDistance to nearest existing observation:")
print(function1_distance)

FUNCTION 1 - EXPLORATION
Selected point:
[0.99997255 0.99984253]

Distance to nearest existing observation:
0.3456413874867929


In [12]:
# Combine all final Week 2 candidate points

final_points = {
    1: function1_point,
    2: week2_candidates[2],
    3: week2_candidates[3],
    4: week2_candidates[4],
    5: week2_candidates[5],
    6: week2_candidates[6],
    7: week2_candidates[7],
    8: week2_candidates[8]
}

In [13]:
for i in range(1, 9):

    point = np.asarray(final_points[i])
    X_existing = data[i].drop(columns="output").values

    # Check all coordinates are between 0 and 1
    in_bounds = np.all((point >= 0) & (point <= 1))

    # Distance to every existing observation
    distances = np.linalg.norm(X_existing - point, axis=1)

    # Closest existing observation
    min_distance = distances.min()

    # Check whether it duplicates an existing point
    duplicate = np.any(np.all(np.isclose(X_existing, point), axis=1))

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print("Selected point:")
    print(np.round(point, 6))

    print(f"In bounds: {in_bounds}")
    print(f"Duplicate existing point: {duplicate}")
    print(f"Distance to nearest observation: {min_distance:.6f}")

    # Additional selection check
    if i == 1:
        expected_index = np.argmax(nearest_distance)
        selection_correct = np.allclose(
            point,
            X_candidates_f1[expected_index]
        )
        print(f"Maximum-distance selection correct: {selection_correct}")

    else:
        gp = models_week2[i]

        # Use the candidates originally generated for this function
        # We need to recreate the same candidate sequence below
        print("Selection method: UCB")

    print()

FUNCTION 1
Selected point:
[0.999973 0.999843]
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.345641
Maximum-distance selection correct: True

FUNCTION 2
Selected point:
[0.655286 0.929896]
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.047468
Selection method: UCB

FUNCTION 3
Selected point:
[0.39753  0.090477 0.447319]
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.291689
Selection method: UCB

FUNCTION 4
Selected point:
[0.480756 0.474    0.229059 0.429178]
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.160930
Selection method: UCB

FUNCTION 5
Selected point:
[0.514842 0.856105 0.961506 0.998492]
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.165759
Selection method: UCB

FUNCTION 6
Selected point:
[0.246549 0.344645 0.479525 0.733976 0.025332]
In bounds: True
Duplicate existing point: False
Distance to ne